# 03 — API: Flight Arrivals
This notebook uses the AeroDataBox API (via RapidAPI) to:
1. Find airports within 50km of each city (**airports** table)
2. Fetch today's incoming flight arrivals for those airports (**flight_arrivals** table)

Flight arrivals are a strong proxy for tourist and business traveller inflow — a key driver of e-scooter demand near airports and city centres.

**Source:** [AeroDataBox via RapidAPI](https://rapidapi.com/aedbx-aedbx/api/aerodatabox)  
**Target tables:** `airports`, `flight_arrivals`  
**Run frequency:** Daily

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import requests
import getpass
from datetime import datetime, timezone, timedelta

---
## 2. Credentials

In [ ]:
password       = getpass.getpass("MySQL password: ")
rapidapi_key   = getpass.getpass("RapidAPI key: ")

schema = "gans_db"
host   = "127.0.0.1"
user   = "root"
port   = 3306

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}"

RAPIDAPI_HEADERS = {
    "x-rapidapi-key":  rapidapi_key,
    "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
    "Content-Type":    "application/json"
}

---
## 3. Extract Airports Near Each City
Searches for airports within a 50km radius of each city's coordinates.
Only airports with active flight data (`withFlightInfoOnly=true`) are returned.

In [ ]:
cities_df = pd.read_sql("SELECT city_id, city_name, latitude, longitude FROM cities", con=connection_string)

def get_airports_for_city(city_id, lat, lon):
    """Find airports within 50km of given coordinates."""
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    params = {
        "lat":                lat,
        "lon":                lon,
        "radiusKm":           "50",
        "limit":              "10",
        "withFlightInfoOnly": "true"
    }
    response = requests.get(url, headers=RAPIDAPI_HEADERS, params=params)

    if response.status_code != 200:
        print(f"[ERROR] Airport lookup failed — status {response.status_code}")
        return []

    airports = []
    for item in response.json().get("items", []):
        airports.append({
            "icao":         item.get("icao"),
            "iata":         item.get("iata"),
            "airport_name": item.get("name"),
            "city_id":      city_id
        })
    return airports


airport_records = []
for _, city in cities_df.iterrows():
    results = get_airports_for_city(city["city_id"], city["latitude"], city["longitude"])
    airport_records.extend(results)
    print(f"{city['city_name']}: found {len(results)} airport(s)")

airports_df = pd.DataFrame(airport_records).dropna(subset=["icao"])
airports_df

---
## 4. Load Airports to MySQL
This table only needs to be populated once (or when adding new cities).

In [ ]:
rows_inserted = airports_df.to_sql(
    "airports",
    if_exists="append",
    con=connection_string,
    index=False
)
print(f"Inserted {rows_inserted} rows into `airports`.")

---
## 5. Extract Flight Arrivals
Fetches today's arrivals for each airport. The API accepts a time window — we use midnight-to-midnight local time (UTC).

Flight data helps anticipate demand spikes: budget airline arrivals correlate strongly with tourist demand for scooters near city centres and landmarks.

In [ ]:
# Read airports back from DB to get confirmed ICAOs
airports_df = pd.read_sql("SELECT icao FROM airports", con=connection_string)

retrieved_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

# Build today's time window (UTC)
today      = datetime.now(timezone.utc).date()
window_from = f"{today}T00:00"
window_to   = f"{today}T23:59"

def get_arrivals(icao):
    """Fetch today's scheduled arrivals for a given airport ICAO code."""
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{icao}/{window_from}/{window_to}"
    params = {
        "withLeg":          "true",
        "direction":        "Arrival",
        "withCancelled":    "false",
        "withCodeshared":   "false",
        "withPrivate":      "false"
    }
    response = requests.get(url, headers=RAPIDAPI_HEADERS, params=params)

    if response.status_code != 200:
        print(f"[ERROR] Arrivals for {icao} — status {response.status_code}")
        return []

    arrivals = []
    for flight in response.json().get("arrivals", []):
        arrivals.append({
            "icao":             icao,
            "flight_number":    flight.get("number"),
            "scheduled_arrival": flight.get("movement", {}).get("scheduledTime", {}).get("utc"),
            "origin_city":      flight.get("movement", {}).get("airport", {}).get("municipalityName"),
            "origin_country":   flight.get("movement", {}).get("airport", {}).get("countryCode"),
            "terminal":         flight.get("movement", {}).get("terminal"),
            "retrieved_at":     retrieved_at
        })
    return arrivals


arrival_records = []
for icao in airports_df["icao"]:
    results = get_arrivals(icao)
    arrival_records.extend(results)
    print(f"{icao}: {len(results)} arrivals found")

arrivals_df = pd.DataFrame(arrival_records)
print(f"\nTotal: {len(arrivals_df)} arrival records")
arrivals_df.head()

---
## 6. Transform — Parse Timestamps

In [ ]:
arrivals_df["scheduled_arrival"] = pd.to_datetime(arrivals_df["scheduled_arrival"], utc=True, errors="coerce")
arrivals_df["retrieved_at"]      = pd.to_datetime(arrivals_df["retrieved_at"])

# Drop rows where we couldn't parse an arrival time
arrivals_df = arrivals_df.dropna(subset=["scheduled_arrival", "flight_number"])

print(f"{len(arrivals_df)} clean records ready to load.")
arrivals_df.dtypes

---
## 7. Load — Push to MySQL

In [ ]:
rows_inserted = arrivals_df.to_sql(
    "flight_arrivals",
    if_exists="append",
    con=connection_string,
    index=False
)
print(f"Inserted {rows_inserted} rows into `flight_arrivals`.")